# Learned reranker from scratch
BM25 first stage, query-doc features, then pointwise / pairwise / MLP rerankers trained on labeled pairs and evaluated on held-out topics.

## 1. Synthetic dataset (seeded) with graded labels and a topic-level split

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
from data import make_dataset
ds = make_dataset(seed=42)
print(len(ds['docs']), 'docs |', len(ds['split']['train']), 'train queries |', len(ds['split']['test']), 'test queries')
for k in ['t00_answer', 't00_partial', 't00_stuffed']:
    print(k, ds['docs'][k])
print(ds['queries'][0])

## 2. First stage (BM25 top-20) and features
The stuffed doc repeats one query word, so BM25 likes it. The `coverage` and `max_qtf` features expose it.

In [ ]:
from features import FirstStage, build_pairs, FEATURES
fs = FirstStage(ds['docs'], ds['background'])
train = build_pairs(fs, ds['split']['train'], 20); test = build_pairs(fs, ds['split']['test'], 20)
g = train[0]
print(g['query'], '| first-stage top-5:', g['cands'][:5], '| labels:', g['y'][:5])
print(dict(zip(FEATURES, np.round(g['X'][0], 3))))

## 3. Train three rerankers

In [ ]:
from rerankers import PointwiseLR, PairwiseLR, PointwiseMLP, rerank
models = [PointwiseLR().fit(train), PairwiseLR().fit(train), PointwiseMLP().fit(train)]
for mdl in models:
    print(f'{mdl.name:14s} loss {mdl.loss[0]:.3f} -> {mdl.loss[-1]:.3f}')
print('pairwise weights:', dict(zip(FEATURES, np.round(models[1].w, 2))))

## 4. Before vs after on held-out topics

In [ ]:
from metrics import evaluate
def ev(groups, fn):
    return round(float(np.mean([evaluate(fn(g), g['rel'])['ndcg@5'] for g in groups])), 3)
print('BM25 order   test nDCG@5', ev(test, lambda g: g['cands']))
for mdl in models:
    print(f'{mdl.name:12s} test nDCG@5', ev(test, lambda g: rerank(mdl, g)))
print('oracle       test nDCG@5', ev(test, lambda g: [g['cands'][i] for i in np.argsort(-g['y'], kind='stable')]))

## 5. Full smoke run
`python run_smoke.py` rewrites `results/` (RESULTS.md, metrics.json, JSON.shot, SVG plots).